<a href="https://colab.research.google.com/github/wallnerlab/afsample3/blob/main/AFsample3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Step 1: Copy AF3 weights from your Google drive, assumes the weights are in the root of your google drive
# Copies AF3 data from Google Drive to local Colab storage
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p alphafold3_data/params
!rsync -Parv drive/MyDrive/af3.bin alphafold3_data/params/af3.bin

In [ ]:
#@title Step 2: Install dependencies by clicking the play button. It will take ~13min
import os
from sys import version_info, path
python_version = f"{version_info.major}.{version_info.minor}"
PYTHON_VERSION = python_version


#os.system("git clone -b notebook https://github.com/clami66/AF_unmasked.git")

if not os.path.isfile("COLABFOLD_READY"):
  print("installing colabfold, to get MMseq2 alignments...")
  os.system("pip install -q --no-warn-conflicts 'colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold'")
 # if os.environ.get('TPU_NAME', False) != False:
 #   os.system("pip uninstall -y jax jaxlib")
 #   os.system("pip install --no-warn-conflicts --upgrade dm-haiku==0.0.10 'jax[cuda12_pip]'==0.3.25 -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html")
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabfold colabfold")
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/alphafold alphafold")
  os.system("touch COLABFOLD_READY")

if not os.path.isfile("AFSAMPLE3_READY"):
  print('Installing AFsample3...')
  !git clone https://github.com/wallnerlab/afsample3
  !cd afsample3;pip3 install .;
  !build_data
  # fake the databases
  !touch alphafold3_data/bfd-first_non_consensus_sequences.fasta
  !touch alphafold3_data/mgy_clusters_2022_05.fa
  !touch alphafold3_data/uniprot_all_2021_04.fa
  !touch alphafold3_data/uniref90_2022_05.fa
  !touch alphafold3_data/nt_rna_2023_02_23_clust_seq_id_90_cov_80_rep_seq.fasta
  !touch alphafold3_data/rfam_14_9_clust_seq_id_90_cov_80_rep_seq.fasta
  !touch alphafold3_data/rnacentral_active_seq_id_90_cov_80_linclust.fasta
  !touch alphafold3_data/mmcif_files
  !touch alphafold3_data/pdb_seqres_2022_09_28.fasta
  !touch AFSAMPLE3_READY

#if not os.path.isfile("CONDA_READY"):
#  print("installing conda...")
#  os.system("wget -qnc https://github.com/conda-forge/miniforge/releases/latest/download/Mambaforge-Linux-x86_64.sh")
#  os.system("bash Mambaforge-Linux-x86_64.sh -bfp /usr/local")
#  os.system("mamba config --set auto_update_conda false")
#  os.system("touch CONDA_READY")

#print("installing conda packages...")
#!mamba install -y -c conda-forge -c bioconda hmmer kalign2=2.04 hhsuite=3.3.0 &> /dev/null
#!pip install logomaker
#!pip install pymsaviz
#!pip install dash-bio
#!pip install alv



import pickle
import shutil
import importlib_metadata
from pathlib import Path
from string import ascii_uppercase, ascii_lowercase
ascii_upperlower = ascii_uppercase + ascii_lowercase

import ipywidgets as widgets
from google.colab import files


from colabfold.batch import get_msa_and_templates
from colabfold.utils import DEFAULT_API_SERVER, get_commit

### Some functions...






In [ ]:
!ls models/test

In [ ]:
#@title Step 2 Upload Alphafold3 weights (af3.bin) - can be slow...
from google.colab import files
uploaded = files.upload()

In [ ]:


#@title Step 3a Upload a protein sequence

jobname = "test" #@param {type:"string"}
jobname = jobname.replace(" ","_")


#@markdown  Paste in your amino acid sequence:
protein_seq = "MKVFIGADHRGFELKEKIAKWLFEMKYDFLDVGAQNLESGDNFTKYTSEVASLVAANENSRGVVLCGSGV GADIAANKFDGARAAIGKSAKQIKEGREDDDMNILVLAADYTSERAKPCLLHF" #@param {type:"string"}
protein_seq=protein_seq.replace(" ","")

with open('test.fa','w') as f:
  f.write(f'>test\n{protein_seq}')


#@markdown
num_seeds=11 #@param

In [ ]:
!ls

In [ ]:
#@title Step 3b Generate a MSA for your protein sequence using MMseqs2
print("Querying ColabFold's MSA server")
jobname='test'
out_dir=Path('.')
target_sequences = [protein_seq]
msa_lines = None
msa_mode="mmseqs2_uniref" #@param ["mmseqs2_uniref", "mmseqs2_uniref_env"]
use_templates = False
custom_template_path = None
pair_mode = "unpaired"
pairing_strategy = "greedy"
host_url = DEFAULT_API_SERVER
version = importlib_metadata.version("colabfold")
commit = get_commit()
if commit:
    version += f" ({commit})"
user_agent = f"colabfold/{version}"

unpaired_msa, paired_msa, query_seqs_unique, query_seqs_cardinality, template_features = get_msa_and_templates(jobname, target_sequences, msa_lines, out_dir, msa_mode, use_templates,
                        custom_template_path, pair_mode, pairing_strategy, host_url, user_agent)
msas=unpaired_msa[0].splitlines()
out_dir = Path(f"{jobname}")
out_dir.mkdir(parents=True, exist_ok=True)
out_dir.joinpath(f"alignment.a3m").write_text(unpaired_msa[0])

with open('alignment_fixed.a3m','w') as f:
  for msa_line in msas:
    if msa_line.startswith('>'):
      f.write(msa_line)
    else:
      for char in msa_line:
        if not char.islower():
          f.write(char)
    f.write("\n")

In [ ]:
os.system('pip3 install -r afsample3/dev-requirements.txt')
!pip3 install -r afsample3/dev-requirements.txt

In [ ]:
!echo {num_seeds}

In [ ]:
#@title Generate AFsample3 ensemble

%env XLA_FLAGS=--xla_disable_hlo_passes=custom-kernel-fusion-rewriter
!python afsample3/prepare_json.py --fasta test.fa --msa alignment_fixed.a3m
!python afsample3/run_alphafold.py --db_dir alphafold3_data/ --model_dir alphafold3_data/params/ --flash_attention_implementation=xla --output_dir=models/ --run_inference=True --msa_rand_fraction 0.4 --json_path test.fa.json --num_seeds {num_seeds}





In [ ]:
#@title Show prediction
import py3Dmol
import matplotlib.pyplot as plt
from colabfold.colabfold import plot_plddt_legend
from colabfold.colabfold import pymol_color_list, alphabet_list

#rank_num = 0 #@param ["0", "1", "2", "3", "4", "5"] {type:"raw"}
color = "lDDT" #@param ["chain", "lDDT", "rainbow"]
show_sidechains = False #@param {type:"boolean"}
show_mainchains = False #@param {type:"boolean"}

def show_pdb(pdb_file, extension, show_sidechains=False, show_mainchains=False, color="lDDT"):
  view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js',)
  view.addModel(open(pdb_file,'r').read(), extension)

  if color == "lDDT":
    view.setStyle({'cartoon': {'colorscheme': {'prop':'b','gradient': 'roygb','min':50,'max':90}}})
  elif color == "rainbow":
    view.setStyle({'cartoon': {'color':'spectrum'}})
  elif color == "chain":
    chains = len(target_sequences) + 1
    for n,chain,color in zip(range(chains),alphabet_list,pymol_color_list):
       view.setStyle({'chain':chain},{'cartoon': {'color':color}})

  if show_sidechains:
    BB = ['C','O','N']
    view.addStyle({'and':[{'resn':["GLY","PRO"],'invert':True},{'atom':BB,'invert':True}]},
                        {'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
    view.addStyle({'and':[{'resn':"GLY"},{'atom':'CA'}]},
                        {'sphere':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
    view.addStyle({'and':[{'resn':"PRO"},{'atom':['C','O'],'invert':True}]},
                        {'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
  if show_mainchains:
    BB = ['C','O','N','CA']
    view.addStyle({'atom':BB},{'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})

  view.zoomTo()
  return view

prediction_pdb = f"models/{jobname}/{jobname}_model.cif"
#template_pdb = f"{mmcif_path}/0000.cif"

#print("Template")
#show_pdb(template_pdb, "cif", show_sidechains, show_mainchains, color).show()
print(f"Prediction {prediction_pdb}")
show_pdb(prediction_pdb, "cif", show_sidechains, show_mainchains, color).show()

In [ ]:
#@title Download results
#exclude_pickles = True #@param {type:"boolean"}
#extra_zip_flags = "-x '*.pkl'" if exclude_pickles else ""

save_to_google_drive = False #@param {type:"boolean"}

def get_unique_filename(path):
    if not os.path.exists(path):
        return path
    base, ext = os.path.splitext(path)
    i = 1
    while os.path.exists(f"{base}_{i}{ext}"):
        i += 1
    return f"{base}_{i}{ext}"

results_zip = f"{jobname}_AFsample3.zip"
os.system(f"zip -r {results_zip} models/{jobname}/")

files.download(results_zip)
if save_to_google_drive:
  drive_path = f"/content/drive/MyDrive/{results_zip}"
  drive_path = get_unique_filename(drive_path)
  shutil.copy(results_zip, drive_path) #f"/content/drive/MyDrive/{results_zip}")

  print(f"Uploaded {results_zip} to Google Drive as {drive_path}")

In [ ]:
!ls /